In [2]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import cg, spsolve
from scipy.linalg import solve_triangular
import time

def create_2d_poisson_matrix(n):
    """
    Create the discrete 2D Poisson matrix using finite differences
    on an n×n interior grid (total unknowns = n²)
    
    The 2D Poisson equation: ∂²u/∂x² + ∂²u/∂y² = f
    discretized with centered differences gives the 5-point stencil
    """
    N = n * n  # Total number of unknowns
    
    # Create the matrix using the 5-point stencil
    # Main diagonal: 4's
    main_diag = 4 * np.ones(N)
    
    # Off-diagonals: -1's
    # Left/right neighbors (same row in grid)
    off_diag = -np.ones(N - 1)
    # Remove connections at row boundaries
    off_diag[n-1::n] = 0
    
    # Up/down neighbors (adjacent rows in grid)
    up_down_diag = -np.ones(N - n)
    
    # Build sparse matrix efficiently
    diagonals = [main_diag, off_diag, off_diag, up_down_diag, up_down_diag]
    offsets = [0, -1, 1, -n, n]
    
    A = sp.diags(diagonals, offsets, shape=(N, N), format='csr')
    
    return A

def create_right_hand_side(n):
    """
    Create a right-hand side vector for the 2D Poisson equation
    This represents the source term f(x,y)
    """
    N = n * n
    x = np.linspace(0, 1, n+2)[1:-1]  # Interior points
    y = np.linspace(0, 1, n+2)[1:-1]
    
    X, Y = np.meshgrid(x, y)
    
    # Example source: f(x,y) = sin(π*x) * sin(π*y)
    f = np.sin(np.pi * X) * np.sin(np.pi * Y)
    
    # Flatten to vector (column-major order matches our matrix structure)
    b = f.flatten()
    
    return b

def solve_with_cholesky(A, b):
    """
    Solve using sparse Cholesky factorization
    """
    from sksparse.cholmod import cholesky
    
    start = time.time()
    
    # Compute Cholesky factorization
    factor = cholesky(A)
    
    # Solve using the factorization
    x = factor(b)
    
    elapsed = time.time() - start
    
    return x, elapsed

def solve_with_conjugate_gradient(A, b, rtol=1e-10):
    """
    Solve using Conjugate Gradient
    """
    start = time.time()
    
    # Solve with CG
    x, info = cg(A, b, rtol=rtol, atol=0)
    
    elapsed = time.time() - start
    
    if info > 0:
        print(f"CG convergence warning: {info} iterations without convergence")
    elif info < 0:
        print(f"CG error: illegal input or breakdown")
    
    return x, elapsed

def compute_residual(A, x, b):
    """
    Compute the relative residual ||Ax - b|| / ||b||
    """
    residual = A @ x - b
    return np.linalg.norm(residual) / np.linalg.norm(b)

# Main comparison
if __name__ == "__main__":
    # Grid sizes to test
    # n=200 gives 40,000 unknowns
    # n=300 gives 90,000 unknowns
    # n=400 gives 160,000 unknowns
    # n=500 gives 250,000 unknowns
    
    grid_sizes = [200, 300, 400, 500]
    
    print("2D Poisson Equation Solver Comparison")
    print("=" * 70)
    print()
    
    for n in grid_sizes:
        N = n * n
        print(f"Grid size: {n}×{n} (Total unknowns: {N:,})")
        print("-" * 70)
        
        # Create the system
        print("Building matrix and RHS...")
        A = create_2d_poisson_matrix(n)
        b = create_right_hand_side(n)
        
        nnz = A.nnz
        density = nnz / (N * N) * 100
        print(f"Matrix: {N}×{N}, Non-zeros: {nnz:,}, Density: {density:.3f}%")
        print()
        
        # Solve with Cholesky
        try:
            print("Solving with Sparse Cholesky...")
            x_chol, time_chol = solve_with_cholesky(A, b)
            res_chol = compute_residual(A, x_chol, b)
            print(f"  Time: {time_chol:.4f} seconds")
            print(f"  Relative residual: {res_chol:.2e}")
        except ImportError:
            print("  scikit-sparse not available, using spsolve instead...")
            start = time.time()
            x_chol = spsolve(A, b)
            time_chol = time.time() - start
            res_chol = compute_residual(A, x_chol, b)
            print(f"  Time: {time_chol:.4f} seconds")
            print(f"  Relative residual: {res_chol:.2e}")
        
        print()
        
        # Solve with Conjugate Gradient
        print("Solving with Conjugate Gradient...")
        x_cg, time_cg = solve_with_conjugate_gradient(A, b)
        res_cg = compute_residual(A, x_cg, b)
        print(f"  Time: {time_cg:.4f} seconds")
        print(f"  Relative residual: {res_cg:.2e}")
        
        print()
        
        # Compare solutions
        diff = np.linalg.norm(x_chol - x_cg) / np.linalg.norm(x_chol)
        print(f"Relative difference between solutions: {diff:.2e}")
        
        # Summary
        if time_chol < time_cg:
            speedup = time_cg / time_chol
            print(f"Cholesky is {speedup:.2f}× faster than CG")
        else:
            speedup = time_chol / time_cg
            print(f"CG is {speedup:.2f}× faster than Cholesky")
        
        print()
        print("=" * 70)
        print()

2D Poisson Equation Solver Comparison

Grid size: 200×200 (Total unknowns: 40,000)
----------------------------------------------------------------------
Building matrix and RHS...
Matrix: 40000×40000, Non-zeros: 199,200, Density: 0.012%

Solving with Sparse Cholesky...
  scikit-sparse not available, using spsolve instead...
  Time: 0.0810 seconds
  Relative residual: 1.10e-12

Solving with Conjugate Gradient...
  Time: 0.0003 seconds
  Relative residual: 1.06e-12

Relative difference between solutions: 5.69e-14
CG is 288.14× faster than Cholesky


Grid size: 300×300 (Total unknowns: 90,000)
----------------------------------------------------------------------
Building matrix and RHS...
Matrix: 90000×90000, Non-zeros: 448,800, Density: 0.006%

Solving with Sparse Cholesky...
  scikit-sparse not available, using spsolve instead...
  Time: 0.1903 seconds
  Relative residual: 2.53e-12

Solving with Conjugate Gradient...
  Time: 0.0008 seconds
  Relative residual: 2.23e-12

Relative diffe